# 04 — Grad-CAM & Variants

Grad-CAM produces visual explanations for convolutional networks by mapping model gradients back to spatial feature maps.

## Gradient-Weighted Class Activation Maps

Grad-CAM (Selvaraju et al., 2017) computes a class-discriminative heatmap by:
1. Computing the gradient of the class score *y^c* with respect to the feature maps **A^k** of the last convolutional layer
2. Global-average-pooling those gradients to get importance weights *α^c_k = (1/Z)∑∑ ∂y^c/∂A^k_{ij}*
3. Forming a weighted sum of feature maps, ReLU'd to keep positive influences: *L^c_{Grad-CAM} = ReLU(∑_k α^c_k A^k)*

The ReLU discards features that decrease the class score. The resulting map is bilinearly upsampled to the input size and overlaid as a heatmap.

**Grad-CAM++** generalises by computing per-pixel gradient weights (instead of global average pooling), giving better localisation when multiple instances of an object appear.

**Score-CAM** removes the gradient dependency entirely — it uses each feature map as a mask, measures the score change, and uses that as the weight. More stable but slower.

In [ ]:
# Grad-CAM on a simple CNN
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Simple CNN for 28x28 grayscale images
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Generate synthetic data (two classes: circle vs square pattern)
def make_circle_pattern(n=28):
    img = np.zeros((n, n))
    cx, cy = n//2, n//2
    for i in range(n):
        for j in range(n):
            if abs((i-cx)**2 + (j-cy)**2 - (n//4)**2) < 20:
                img[i, j] = 1.0
    return img

def make_square_pattern(n=28):
    img = np.zeros((n, n))
    start, end = n//4, 3*n//4
    img[start:start+2, start:end] = 1.0
    img[end-2:end, start:end] = 1.0
    img[start:end, start:start+2] = 1.0
    img[start:end, end-2:end] = 1.0
    return img

# Build dataset
n_samples = 200
X_data, y_data = [], []
for _ in range(n_samples // 2):
    img = make_circle_pattern() + np.random.normal(0, 0.1, (28, 28))
    X_data.append(img)
    y_data.append(0)
    img = make_square_pattern() + np.random.normal(0, 0.1, (28, 28))
    X_data.append(img)
    y_data.append(1)

X_tensor = torch.tensor(np.array(X_data), dtype=torch.float32).unsqueeze(1)
y_tensor = torch.tensor(y_data)

cnn = SimpleCNN(n_classes=2)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
for epoch in range(20):
    cnn.train()
    logits = cnn(X_tensor)
    loss = nn.CrossEntropyLoss()(logits, y_tensor)
    opt.zero_grad(); loss.backward(); opt.step()

cnn.eval()
with torch.no_grad():
    preds = cnn(X_tensor).argmax(1)
    acc = (preds == y_tensor).float().mean()
print(f'Training accuracy: {acc:.3f}')

In [ ]:
# Grad-CAM implementation
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def compute(self, x, class_idx=None):
        x = x.unsqueeze(0).requires_grad_(True)
        logits = self.model(x)

        if class_idx is None:
            class_idx = logits.argmax(1).item()

        self.model.zero_grad()
        logits[0, class_idx].backward()

        # Global average pool gradients over spatial dims
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam = torch.relu(cam).squeeze().numpy()

        # Normalise
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam, class_idx

# Apply Grad-CAM to last conv layer
target_layer = cnn.features[3]  # second conv layer
gradcam = GradCAM(cnn, target_layer)

# Explain a circle and a square
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, idx in enumerate([0, 1]):
    x = X_tensor[idx]
    cam, pred_class = gradcam.compute(x)

    # Original image
    axes[row, 0].imshow(x.squeeze().numpy(), cmap='gray')
    axes[row, 0].set_title(f'Input (class={y_data[idx]})')
    axes[row, 0].axis('off')

    # Grad-CAM heatmap overlaid
    import torch.nn.functional as F
    cam_up = F.interpolate(torch.tensor(cam).unsqueeze(0).unsqueeze(0),
                           size=(28, 28), mode='bilinear').squeeze().numpy()
    axes[row, 1].imshow(x.squeeze().numpy(), cmap='gray')
    axes[row, 1].imshow(cam_up, cmap='jet', alpha=0.5)
    axes[row, 1].set_title(f'Grad-CAM (pred={pred_class})')
    axes[row, 1].axis('off')

plt.suptitle('Grad-CAM Visualisations')
plt.tight_layout()
plt.savefig('/tmp/gradcam.png', dpi=100, bbox_inches='tight')
plt.show()
print('Grad-CAM visualisation saved')